In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("image_paths.csv")

In [3]:
df.head()

,class_path,file_path
0,A,ISL_Dataset\A\A (1).jpg
1,A,ISL_Dataset\A\A (10).jpg
2,A,ISL_Dataset\A\A (11).jpg
3,A,ISL_Dataset\A\A (12).jpg
4,A,ISL_Dataset\A\A (13).jpg


In [4]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['class_path'] = le.fit_transform(df['class_path'])

In [5]:
from sklearn.model_selection import train_test_split

X = df['file_path']
y = df['class_path']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [6]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [7]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(X_val.shape, y_val.shape)

(491,) (491,)
(106,) (106,)
(105,) (105,)


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SignDetection(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['file_path']).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['class_path'], dtype=torch.long)
        return image, label


train_df = pd.DataFrame({"file_path": X_train, "class_path": y_train}).reset_index(drop=True)
val_df   = pd.DataFrame({"file_path": X_val, "class_path": y_val}).reset_index(drop=True)
test_df  = pd.DataFrame({"file_path": X_test, "class_path": y_test}).reset_index(drop=True)


In [9]:
from torchvision import transforms

# ImageNet mean/std — required when using any ImageNet-pretrained model (ResNet, MobileNet, etc.)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),          # ResNet's expected input size
    transforms.RandomHorizontalFlip(p=0.5), # augmentation — only on train
    transforms.RandomRotation(10),          # small rotation, hand shapes shouldn't flip upside down
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),                  # PIL Image -> tensor, scales pixels to [0,1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [10]:
train_dataset = SignDetection(train_df, transform=train_transform)
val_dataset   = SignDetection(val_df, transform=val_transform)
test_dataset  = SignDetection(test_df, transform=val_transform)

In [11]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.6.0+cu124
CUDA available: True
Using device: cuda


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
print(f"Device: {device}")

Device: cuda
